# 00 - Competition Intro And Data Setup

This notebook is the first pass through the Kaggle dataset.

Goals:
- load the raw competition files
- check shapes, columns, and date coverage
- inspect missing values
- inspect the target distribution and its outliers

Use this notebook before trying to model anything.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

ROOT = Path.cwd().resolve().parent
DATA_DIR = ROOT / 'data' / 'raw'
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SAMPLE_PATH = DATA_DIR / 'sample_submission.csv'

ROOT, DATA_DIR

In [ ]:
required_files = [TRAIN_PATH, TEST_PATH, SAMPLE_PATH]
missing_files = [path.name for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        'Missing competition files in data/raw: ' + ', '.join(missing_files)
    )

print('All required files are present.')

In [ ]:
date_columns = ['period_start', 'period_end']

train = pd.read_csv(TRAIN_PATH, parse_dates=date_columns)
test = pd.read_csv(TEST_PATH, parse_dates=date_columns)
sample_submission = pd.read_csv(SAMPLE_PATH)

print('train shape:', train.shape)
print('test shape:', test.shape)
print('sample_submission shape:', sample_submission.shape)

print('\nTrain date range:')
print(train['period_start'].min(), 'to', train['period_start'].max())
print('Train forward return end range:')
print(train['period_end'].min(), 'to', train['period_end'].max())

print('\nTest date range:')
print(test['period_start'].min(), 'to', test['period_start'].max())
print('Test holding period end range:')
print(test['period_end'].min(), 'to', test['period_end'].max())

In [ ]:
display(train.head())
display(test.head())

print('Number of train columns:', len(train.columns))
print('Number of test columns:', len(test.columns))

print('\nColumn dtypes:')
display(train.dtypes.sort_index())

In [ ]:
missing_summary = (
    train.isna()
    .mean()
    .sort_values(ascending=False)
    .rename('missing_rate')
    .to_frame()
)
missing_summary['missing_pct'] = 100 * missing_summary['missing_rate']

display(missing_summary.head(20))

plt.figure(figsize=(10, 8))
top_missing = missing_summary.head(20).sort_values('missing_pct')
plt.barh(top_missing.index, top_missing['missing_pct'])
plt.title('Top 20 Train Features By Missingness')
plt.xlabel('Missing percentage')
plt.tight_layout()
plt.show()

In [ ]:
target = train['return_pct']

print(target.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(target, bins=80, ax=axes[0])
axes[0].set_title('Target Distribution: return_pct')
axes[0].set_xlabel('1-year forward return (%)')

sns.boxplot(x=target, ax=axes[1])
axes[1].set_title('Target Boxplot')
axes[1].set_xlabel('1-year forward return (%)')

plt.tight_layout()
plt.show()

In [ ]:
train = train.assign(obs_year=train['period_start'].dt.year)

year_summary = train.groupby('obs_year')['return_pct'].agg(['count', 'mean', 'median', 'std'])
display(year_summary)

plt.figure(figsize=(8, 5))
sns.boxplot(data=train, x='obs_year', y='return_pct')
plt.ylim(train['return_pct'].quantile(0.01), train['return_pct'].quantile(0.99))
plt.title('Target Distribution By Observation Year (1st to 99th pct clipped view)')
plt.xlabel('Observation year')
plt.ylabel('return_pct')
plt.tight_layout()
plt.show()

## What To Notice

- Missing values are part of the competition, not a data-cleaning mistake.
- The target is likely to be heavy-tailed, so RMSE can move a lot from a few names.
- Date-aware validation is mandatory because the test set begins in 2024.

Next: open `01_eda_and_validation_design.ipynb` and design a defensible validation scheme.